# Depth Estimation using Depth Anything

**Student Name:** Lina Mohammed Badr  
**Course:** [Put your course name]  
**Project:** Integration-based AI System  

This notebook demonstrates the implementation and testing of a depth estimation model using a pre-trained deep learning model.


## Objective

The goal of this task is to estimate depth information from a monocular video.

Depth estimation allows the system to understand how far objects are from the camera, which is useful for scene understanding and decision-making.


## Model Description

We use a pre-trained model called **Depth Anything** for monocular depth estimation.

The model has been trained on large-scale datasets and can predict depth from a single image without requiring stereo input.

This notebook focuses on implementation, inference, and testing.


## Setup

Import required libraries.


In [6]:
!pip install opencv-python
!pip install opencv-python torch torchvision transformers matplotlib


  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ---------------------------------------- 0.3/114.6 MB ? eta -:--:--
   ---------------------------------------- 0.3/114.6 MB ? eta -:--:--
   ---------------------------------------- 0.5/114.6 MB 584.2 kB/s eta 0:03:16
   ---------------------------------------- 0.5/114.6 MB 584.2 kB/s eta 0:03:16
   ---------------------------------------- 0.8/114.6 MB 736.9 kB/s eta 0:02:35
   ---------------------------------------- 1.0/114.6 MB 855.2 kB/s eta 0:02:13
   ---------------------------------------- 1.3/114.6 MB 796.8 kB/s eta 0:02:23
    --------------------------------------- 1.6/114.6 MB 876.6 kB/s eta 0:02:09
    --------------------------------------- 1.8/114.6 MB 936.2 kB/s eta 0:02:01
    -------------------------

In [7]:
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoImageProcessor, AutoModelForDepthEstimation


c:\Users\linam\Gradution_Tammakan\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Pre-trained Model

Load the model and move it to GPU if available.


In [8]:
processor = AutoImageProcessor.from_pretrained("LiheYoung/depth-anything-small-hf")
model = AutoModelForDepthEstimation.from_pretrained("LiheYoung/depth-anything-small-hf")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()


c:\Users\linam\Gradution_Tammakan\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\linam\.cache\huggingface\hub\models--LiheYoung--depth-anything-small-hf. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 287/287 [00:00<00:00, 4301.90it/s]


DepthAnythingForDepthEstimation(
  (backbone): Dinov2Backbone(
    (embeddings): Dinov2Embeddings(
      (patch_embeddings): Dinov2PatchEmbeddings(
        (projection): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): Dinov2Encoder(
      (layer): ModuleList(
        (0-11): 12 x Dinov2Layer(
          (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
          (attention): Dinov2Attention(
            (attention): Dinov2SelfAttention(
              (query): Linear(in_features=384, out_features=384, bias=True)
              (key): Linear(in_features=384, out_features=384, bias=True)
              (value): Linear(in_features=384, out_features=384, bias=True)
            )
            (output): Dinov2SelfOutput(
              (dense): Linear(in_features=384, out_features=384, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (layer_scale1): Di

## Depth Prediction Function

This function takes a frame and returns a normalized depth map.


In [9]:
def predict_depth(frame):

    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        predicted_depth = outputs.predicted_depth

    depth = predicted_depth.squeeze().cpu().numpy()
    depth = cv2.resize(depth, (frame.shape[1], frame.shape[0]))

    depth_norm = cv2.normalize(depth, None, 0, 255, cv2.NORM_MINMAX)
    depth_norm = depth_norm.astype(np.uint8)

    return depth_norm


## Video Processing

Process the video frame by frame and save depth output.


In [10]:
def process_video(input_path, output_path):

    cap = cv2.VideoCapture(input_path)

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height), False)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        depth_map = predict_depth(frame)
        out.write(depth_map)

    cap.release()
    out.release()


## Testing

Run the model on a sample video.


In [11]:
input_video = "10_Sec_Clip.mp4"
output_video = "output_depth.mp4"

process_video(input_video, output_video)


## Display Output Video


In [ ]:
from IPython.display import Video
Video("output_depth.mp4", embed=True)


## Depth Distribution Visualization


In [13]:
def plot_depth_histogram(depth_map):
    plt.figure()
    plt.hist(depth_map.flatten(), bins=50)
    plt.title("Depth Value Distribution")
    plt.xlabel("Depth Value")
    plt.ylabel("Frequency")
    plt.show()

cap = cv2.VideoCapture("input.mp4")
ret, frame = cap.read()

if ret:
    depth_map = predict_depth(frame)
    plot_depth_histogram(depth_map)

cap.release()


## Original vs Depth Map


In [14]:
cap = cv2.VideoCapture("input.mp4")
ret, frame = cap.read()

if ret:
    depth_map = predict_depth(frame)
    colored = cv2.applyColorMap(depth_map, cv2.COLORMAP_INFERNO)

    combined = np.hstack((frame, colored))

    plt.figure(figsize=(10,5))
    plt.imshow(cv2.cvtColor(combined, cv2.COLOR_BGR2RGB))
    plt.title("Original vs Depth Map")
    plt.axis('off')
    plt.show()

cap.release()


## Real-Time Depth Estimation

Press ESC to exit.


In [ ]:
def run_realtime_depth():

    cap = cv2.VideoCapture(0)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        depth_map = predict_depth(frame)

        colored = cv2.applyColorMap(depth_map, cv2.COLORMAP_INFERNO)
        combined = np.hstack((frame, colored))

        cv2.imshow("Real-Time Depth", combined)

        if cv2.waitKey(1) & 0xFF == 27:
            break

    cap.release()
    cv2.destroyAllWindows()

run_realtime_depth()
